# Predictive Comparison: Logistic Regression vs Random Forest vs XGBoost

UK RMBS loan-level data. Secondary/contextual content around RQ1 - the interpretability-accuracy trade-off (Breiman's "two cultures"; Lessmann et al. 2015; Bucker et al. 2020), not a standalone research question (`decisions_log.md`, Part B.1). RQ1's inference result does not depend on anything in this notebook.

**Feature set** excludes `InterestRate_AtReference` (present in the inference model) - that feature's threshold was discovered via exploratory analysis on the full sample, so carrying it into a train/test-split comparison would leak test-set information into a derived feature (`decisions_log.md`, Part B.5c). All three models here get identical, leakage-free inputs: `Original_LTV`, `Interest_Rate` (raw, linear), `Loan_Term`, `Income_log`, and the `Property_Type`/`Purpose`/`Income_Verification` dummies.

**Class imbalance** handled via class-weighting (not SMOTE) for all three models - simpler to apply identically and defensibly across different algorithm types, avoids a second imbalance-handling justification stacked on an already extensively-documented pipeline. SMOTE was considered and deferred, not ruled out, if time allows revisiting.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_predict,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    classification_report, confusion_matrix,
)


def youdens_j_threshold(y_true, y_pred_prob):
    """Threshold maximising Youden's J = TPR - FPR (Youden, 1950) - same
    quantity as the KS statistic used for cutoffs in credit scoring
    (Hand & Henley, 1997). Mirrors the threshold logic already used in
    the inference script for consistency across the two scripts."""
    fpr, tpr, thresh = roc_curve(y_true, y_pred_prob)
    return thresh[np.argmax(tpr - fpr)]


from google.colab import drive
drive.mount("/content/drive")

DATA_FOLDER = "/content/drive/MyDrive/Dissertation/data"
RANDOM_STATE = 42


In [ ]:
# Okabe-Ito palette, continued from the cleaning and inference scripts
PALETTE = {
    "primary": "#0072B2",
    "highlight": "#D55E00",
    "secondary": "#009E73",
    "tertiary": "#E69F00",
    "neutral": "#3A3A3A",
}
plt.rcParams.update({
    "axes.edgecolor": PALETTE["neutral"],
    "axes.labelcolor": PALETTE["neutral"],
    "xtick.color": PALETTE["neutral"],
    "ytick.color": PALETTE["neutral"],
    "text.color": PALETTE["neutral"],
    "axes.grid": True,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.5,
    "figure.dpi": 110,
})

FIGURE_COUNTER = {"n": 18}  # continues from the inference script's Figure 18


def styled_figure(title, xlabel, ylabel):
    FIGURE_COUNTER["n"] += 1
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.set_title(f"Figure {FIGURE_COUNTER['n']}: {title}",
                 fontsize=11, fontweight="bold", color=PALETTE["neutral"], pad=12)
    ax.set_xlabel(xlabel, fontsize=9.5)
    ax.set_ylabel(ylabel, fontsize=9.5)
    return fig, ax


## Step 1: Load Data

In [ ]:
# same reproducibility check as the other two scripts
df = pd.read_csv(f"{DATA_FOLDER}/cleaned_dissertation_data.csv", low_memory=False)
print(f"Loaded: {len(df):,} rows, {df.shape[1]} cols, "
      f"default rate = {df['default'].mean():.2%}")
print("Compare against the cleaning script's final printout "
      "(163,348 rows, 61 cols, 1.05%) before proceeding.")

# feature set - deliberately excludes InterestRate_AtReference (leakage,
# see intro cell); identical across all three models
predictors = [
    "Original_LTV", "Interest_Rate", "Loan_Term", "Income_log",
    "PropType_Flat", "PropType_Bungalow", "PropType_Terraced",
    "Purpose_Remortgage", "Purpose_DebtConsolidation", "Purpose_Renovation",
    "Purpose_RemortgageEquityRelease", "Purpose_Other", "IncVerif_Other",
]
X = df[predictors].astype(float)
y = df["default"].astype(int)


## Step 2: Train/Test Split and Class Weighting

80/20 stratified split - reserved for exactly this comparison since the inference model (RQ1) is fit on the full sample instead (`decisions_log.md`, Part B.2).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train):,} rows, {y_train.sum():,} events "
      f"({y_train.mean():.2%})")
print(f"Test: {len(X_test):,} rows, {y_test.sum():,} events "
      f"({y_test.mean():.2%}) - held out untouched until final evaluation")

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nClass weighting: scale_pos_weight (XGBoost) = {scale_pos_weight:.2f}; "
      f"'balanced' (sklearn) applies an equivalent inverse-frequency weight "
      f"for LR/RF.")

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## Step 3a: Tune Logistic Regression

Standardized inside a `Pipeline` - predictors have very different natural scales (`Loan_Term` up to ~600 vs 0/1 dummies), which caused `lbfgs` convergence failures at low regularization (high `C`) when unscaled. Standardizing is harmless for RF/XGBoost (scale-invariant, not applied there) and means LR's coefficients are already on a comparable scale for the Step 5 feature-importance comparison, with no post-hoc rescaling needed.

Tuned on the training set only (inner CV), scored on PR-AUC - more informative than ROC-AUC under this level of imbalance (Saito & Rehmsmeier, 2015, *PLOS ONE* 10(3), e0118432), consistent with the primary metric used throughout the inference script.

In [ ]:
lr_grid = {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100]}
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs")),
])
lr_search = GridSearchCV(
    lr_pipeline, lr_grid, cv=inner_cv, scoring="average_precision", n_jobs=-1,
)
lr_search.fit(X_train, y_train)
print(f"Best LR: C={lr_search.best_params_['clf__C']}, "
      f"CV PR-AUC={lr_search.best_score_:.4f}")


## Step 3b: Tune Random Forest

`max_depth=None` (unconstrained) and `min_samples_leaf=1` measured at ~35s per fit on this data's 130,678 training rows vs ~8s for a constrained config - genuinely expensive, not just a parallelism artifact, and low value for a secondary/contextual comparison. Excluded from the grid rather than left in and hoping `RandomizedSearchCV` avoids drawing them.

`RandomizedSearchCV` (10 iterations) used instead of exhaustive grid search for bounded, predictable runtime. `n_jobs=-1` is set only at the search level (parallelises across CV folds/candidates) - the estimator itself is `n_jobs=1`. Setting both to `-1` causes nested parallelism, oversubscribing CPU cores and making runs slower, not faster - a contributing cause of an earlier 42-minute runtime on this step.

In [ ]:
rf_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, 20],
    "min_samples_leaf": [5, 20, 50],
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1),
    rf_grid, n_iter=10, cv=inner_cv, scoring="average_precision",
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf_search.fit(X_train, y_train)
print(f"Best RF: {rf_search.best_params_}, CV PR-AUC={rf_search.best_score_:.4f}")


## Step 3c: Tune XGBoost

In [ ]:
xgb_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1, 0.3],
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE,
                  eval_metric="logloss", n_jobs=1),
    xgb_grid, n_iter=10, cv=inner_cv, scoring="average_precision",
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
print(f"Best XGBoost: {xgb_search.best_params_}, "
      f"CV PR-AUC={xgb_search.best_score_:.4f}")

models = {
    "Logistic Regression": lr_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_,
    "XGBoost": xgb_search.best_estimator_,
}


## Step 4: Test-Set Evaluation

Final, single evaluation on the held-out test set - touched exactly once, here, per model, using each model's tuned configuration.

In [ ]:
results = {}
for name, model in models.items():
    test_pred = model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, test_pred)
    pr_auc = average_precision_score(y_test, test_pred)
    results[name] = {"pred": test_pred, "roc_auc": roc_auc, "pr_auc": pr_auc}
    print(f"{name}: ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f} "
          f"(base rate={y_test.mean():.4f})")


### ROC and Precision-Recall Curves, All Three Models Overlaid

In [ ]:
fig, ax = styled_figure("ROC Curves: LR vs Random Forest vs XGBoost (Test Set)",
                         "False Positive Rate", "True Positive Rate")
colors = {"Logistic Regression": PALETTE["primary"],
          "Random Forest": PALETTE["secondary"],
          "XGBoost": PALETTE["highlight"]}
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["pred"])
    ax.plot(fpr, tpr, color=colors[name], label=f"{name} (AUC={res['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], color=PALETTE["neutral"], linestyle="--", linewidth=1)
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.savefig(f"{DATA_FOLDER}/fig{FIGURE_COUNTER['n']}_roc_comparison.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = styled_figure("Precision-Recall Curves: LR vs Random Forest vs XGBoost (Test Set)",
                         "Recall", "Precision")
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res["pred"])
    ax.plot(rec, prec, color=colors[name], label=f"{name} (PR-AUC={res['pr_auc']:.3f})")
ax.axhline(y_test.mean(), color=PALETTE["neutral"], linestyle="--", linewidth=1,
           label=f"Base rate = {y_test.mean():.3f}")
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.savefig(f"{DATA_FOLDER}/fig{FIGURE_COUNTER['n']}_pr_comparison.png", bbox_inches="tight")
plt.show()


### Calibration Comparison, All Three Models

Extends the same calibration-checking logic already used in the inference script (Fig 15) to these three models. Tree ensembles are well-documented to sometimes discriminate better (higher AUC) while calibrating worse than logistic regression - checking both properties, not just AUC, directly serves the interpretability-vs-flexibility discussion.

In [ ]:
fig, ax = styled_figure(
    "Calibration Comparison: LR vs Random Forest vs XGBoost (Test Set)",
    "Mean Predicted Probability", "Observed Default Rate"
)
max_val = 0
for name, res in results.items():
    calib_df = pd.DataFrame({"pred": res["pred"], "obs": y_test.values})
    calib_df["decile"] = pd.qcut(calib_df["pred"], q=10, duplicates="drop")
    calib = calib_df.groupby("decile", observed=True).agg(
        mean_pred=("pred", "mean"), mean_obs=("obs", "mean")
    )
    max_val = max(max_val, calib["mean_pred"].max(), calib["mean_obs"].max())
    ax.plot(calib["mean_pred"], calib["mean_obs"], marker="o", color=colors[name],
             label=name)
ax.plot([0, max_val * 1.1], [0, max_val * 1.1], color=PALETTE["neutral"],
        linestyle="--", linewidth=1, label="Perfect calibration")
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.savefig(f"{DATA_FOLDER}/fig{FIGURE_COUNTER['n']}_calibration_comparison.png",
            bbox_inches="tight")
plt.show()


### Classification Reports (Precision/Recall/F1 per Class)

Same tabular format `sklearn`'s `classification_report` produces. Threshold selection stays leakage-safe: Youden's J / KS threshold (Youden, 1950; Hand & Henley, 1997) is derived from each model's **training-set out-of-fold predictions** (`cross_val_predict`, same `inner_cv` used for hyperparameter tuning), never from the test set directly - consistent with the discipline already used in the inference script.

In [ ]:
for name, model in models.items():
    oof_pred = cross_val_predict(
        model, X_train, y_train, cv=inner_cv, method="predict_proba", n_jobs=-1
    )[:, 1]
    threshold = youdens_j_threshold(y_train, oof_pred)
    test_pred_class = (results[name]["pred"] >= threshold).astype(int)
    print(f"\n{name} (threshold={threshold:.4f}):")
    print(classification_report(y_test, test_pred_class,
                                 target_names=["No Default", "Default"], digits=4))
    cm = confusion_matrix(y_test, test_pred_class, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm, index=["Actual: No Default", "Actual: Default"],
        columns=["Predicted: No Default", "Predicted: Default"]
    )
    print(f"Confusion matrix ({name}, test set, n={len(y_test):,}):")
    print(cm_df)


## Step 5: Feature Importance Comparison

LR's coefficients come from the `StandardScaler` pipeline, so they're already on a comparable scale to each other (and, by rank, roughly comparable to tree-based importances) - raw, unscaled LR coefficients wouldn't be directly comparable across predictors with very different natural scales (e.g. `Original_LTV` 0-100 vs a 0/1 dummy). **Rankings, not raw magnitudes, are what's comparable across model types** - the three importance metrics are on different scales by construction.

In [ ]:
lr_coefs = models["Logistic Regression"].named_steps["clf"].coef_[0]
lr_importance = pd.Series(np.abs(lr_coefs), index=predictors).sort_values(ascending=False)

rf_importance = pd.Series(
    models["Random Forest"].feature_importances_, index=predictors
).sort_values(ascending=False)

xgb_importance = pd.Series(
    models["XGBoost"].feature_importances_, index=predictors
).sort_values(ascending=False)

importance_table = pd.DataFrame({
    "LR (|std coef|)": lr_importance,
    "RF (Gini importance)": rf_importance,
    "XGBoost (gain-based)": xgb_importance,
}).round(4)
print(importance_table)

top3_lr = set(lr_importance.head(3).index)
top3_rf = set(rf_importance.head(3).index)
top3_xgb = set(xgb_importance.head(3).index)
print(f"\nTop-3 agreement: LR&RF={len(top3_lr & top3_rf)}/3, "
      f"LR&XGBoost={len(top3_lr & top3_xgb)}/3, "
      f"RF&XGBoost={len(top3_rf & top3_xgb)}/3")


### Feature Importance Chart

Three side-by-side subplots, not one grouped chart - the three importance metrics are on different scales by construction (per the table above), so a single shared-axis grouped bar would misleadingly imply magnitude comparability that doesn't exist. Each subplot sorted by that model's own ranking.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(f"Figure {FIGURE_COUNTER['n'] + 1}: Feature Importance by Model",
             fontsize=11, fontweight="bold", color=PALETTE["neutral"])
FIGURE_COUNTER["n"] += 1
importances = {
    "Logistic Regression": (lr_importance, PALETTE["primary"]),
    "Random Forest": (rf_importance, PALETTE["secondary"]),
    "XGBoost": (xgb_importance, PALETTE["highlight"]),
}
for ax, (name, (imp, color)) in zip(axes, importances.items()):
    imp_sorted = imp.sort_values()
    ax.barh(imp_sorted.index, imp_sorted.values, color=color, edgecolor="white")
    ax.set_title(name, fontsize=9.5, color=PALETTE["neutral"])
    ax.tick_params(axis="y", labelsize=7.5)
    ax.tick_params(axis="x", labelsize=7.5)
plt.tight_layout()
plt.savefig(f"{DATA_FOLDER}/fig{FIGURE_COUNTER['n']}_feature_importance.png",
            bbox_inches="tight")
plt.show()


## Export Result Tables

In [ ]:
comparison_summary = pd.DataFrame({
    name: {"ROC-AUC": res["roc_auc"], "PR-AUC": res["pr_auc"]}
    for name, res in results.items()
}).T
comparison_summary.round(4).to_csv(f"{DATA_FOLDER}/model_comparison_summary.csv")
importance_table.to_csv(f"{DATA_FOLDER}/feature_importance_comparison.csv")
print(f"Saved comparison summary and feature importance tables to {DATA_FOLDER}/")
